# SPHEREx | Motion and brightness-change detector
CPU training on Planet X v1.2.0. Attach the extracted dataset or its ZIP in Kaggle, then Run All. All model code is included in this notebook. The results directory contains the fitted model, scores, recovery tables and real-image candidates. Python 3.11 and requirements.txt reproduce the local training environment.

In [1]:
from pathlib import Path
import os, zipfile, shutil
# Optionally set DATASET_PATH to the extracted release folder or original ZIP.
location = os.environ.get('DATASET_PATH')
if location:
    dataset = Path(location)
else:
    roots = [Path('data'), Path('/kaggle/input'), Path.home()/'Downloads']
    datasets = [p.parent for d in roots if d.exists() for p in d.glob('**/index.parquet') if (p.parent/'spherex/frames.parquet').exists()]
    archives = [p for d in roots if d.exists() for p in d.glob('**/planetx-dataset-v1.2.0.zip')]
    if not datasets and not archives:
        raise FileNotFoundError('Attach planetx-dataset-v1.2.0.zip or set DATASET_PATH.')
    dataset = (datasets or archives)[0]
if dataset.is_file():
    dest = Path('data/planetx-dataset-v1.2.0')
    with zipfile.ZipFile(dataset) as archive:
        for entry in archive.infolist():
            parts = Path(entry.filename).parts
            if len(parts)<2 or parts[0]!='planetx-dataset-v1.2.0' or entry.is_dir():
                continue
            if parts[1] not in ('index.parquet','spherex'):
                continue
            target = dest.joinpath(*parts[1:]).resolve()
            if not target.is_relative_to(dest.resolve()):
                raise ValueError(entry.filename)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as src, target.open('wb') as dst:
                shutil.copyfileobj(src,dst)
    dataset = dest
print(dataset)

data\planetx-dataset-v1.2.0


## Find image peaks and train
The detector searches aligned images without catalogue coordinates. Injection truth supplies supervised labels after candidate extraction. Validation selects a review threshold. The eight test fields are held out.

In [2]:
"""Blind peak proposals and supervised SPHEREx change classification."""
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve, average_precision_score

FEATURES = ["peak", "flux_z", "reference_z", "diff_z", "dip", "rise",
            "centroid_shift", "flux_contrast", "valid_fraction", "wave_gap"]
CLASSES = ["background", "motion", "variable", "transient"]
KEY = ["realization_id", "frame_index", "obj_id"]


def proposals(path, frames, injected):
    """Return peaks in calibrated, aligned images without access to object labels."""
    with np.load(path) as stack:
        ids = frames.index.to_numpy()
        if injected and not np.array_equal(ids, stack["frame_index"]):
            raise ValueError("Frame metadata does not match image order")
        image, valid = stack["image"], stack["valid"].astype(bool)
        if not injected:
            image, valid = image[ids], valid[ids]
    image = np.where(valid, image, np.nan)
    times = frames.mjd_mid.to_numpy()
    wave = frames.wave_center_um.to_numpy()
    detectors = frames.detector.to_numpy()
    yy, xx = np.mgrid[-2:3, -2:3]
    result = []
    for i, frame_id in enumerate(ids):
        refs = np.flatnonzero((detectors == detectors[i]) & (abs(times - times[i]) > 1))
        if len(refs) < 2:
            continue
        refs = refs[np.argsort(abs(np.log(wave[refs] / wave[i])))[:3]]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ref = np.nanmedian(image[refs], axis=0)
        good = np.isfinite(image[i]) & np.isfinite(ref)
        current = np.where(good, image[i], 0)
        reference = np.where(good, ref, 0)
        diff = gaussian_filter(current - reference, 0.85)
        noise = max(1e-8, 1.4826 * np.median(abs(diff[good] - np.median(diff[good])))) if good.any() else 1
        z = diff / noise
        strength = abs(z)
        peaks = good & (strength >= 3.5) & (strength == maximum_filter(strength, 5))
        peaks[:2] = peaks[-2:] = False
        peaks[:, :2] = peaks[:, -2:] = False
        y, x = np.where(peaks)
        if not len(x):
            continue
        region = (y[:, None, None] + yy, x[:, None, None] + xx)
        a, b, d = current[region], reference[region], z[region]
        def centroid(patch):
            weights = np.maximum(patch - patch.min((1, 2), keepdims=True), 0)
            total = np.maximum(weights.sum((1, 2)), 1e-8)
            return np.array([(weights * axis).sum((1, 2)) / total for axis in (xx, yy)])
        signal = max(float(frames.pixel_noise.iloc[i]), 1e-8)
        flux_a, flux_b = a.sum((1, 2)), b.sum((1, 2))
        features = [strength[y, x], flux_a / signal, flux_b / signal, z[y, x],
                    d.min((1, 2)), d.max((1, 2)), np.linalg.norm(centroid(a)-centroid(b), axis=0),
                    (flux_a-flux_b)/(abs(flux_a)+abs(flux_b)+signal), good[region].mean((1, 2)),
                    np.full(len(x), np.max(abs(np.log(wave[refs]/wave[i]))))]
        table = pd.DataFrame(np.column_stack(features), columns=FEATURES)
        table["frame_index"], table["x"], table["y"] = frame_id, x, y
        result.append(table)
    return pd.concat(result, ignore_index=True) if result else pd.DataFrame(columns=FEATURES+["frame_index","x","y"])


def prepare(root):
    ix = pd.read_parquet(root/"index.parquet").query("mission == 'spherex'")
    frames = pd.read_parquet(root/"spherex/frames.parquet").query("qc_pass")
    objects = pd.read_parquet(root/"spherex/injected_objects.parquet")
    truth = pd.read_parquet(root/"spherex/injected_truth.parquet").merge(
        objects[["realization_id","obj_id","class","is_mover"]], on=["realization_id","obj_id"], validate="many_to_one")
    truth["label"] = np.select([truth["class"].eq("variable"), truth["class"].eq("transient"), truth.is_mover],
                               ["variable","transient","motion"], default="background")
    eligible = truth.query("present and in_grid and masked_frac < 0.5 and not saturated").copy()
    return ix, frames, eligible


def collect(root, ix, frames, eligible):
    samples = []
    for row in ix.query("sample_type == 'injected'").itertuples():
        meta = frames[frames.stack_id == row.stack_id].set_index("frame_index").sort_index()
        candidates = proposals(root/row.path, meta, True)
        candidates["obj_id"], candidates["label"] = -1, "background"
        labels = eligible[eligible.realization_id == row.realization_id]
        for fid, group in candidates.groupby("frame_index"):
            stars = labels[labels.frame_index == fid]
            if stars.empty:
                continue
            dist, idx = cKDTree(stars[["x","y"]]).query(group[["x","y"]])
            near = dist <= 2
            candidates.loc[group.index[near], ["obj_id","label"]] = stars.iloc[idx[near]][["obj_id","label"]].to_numpy()
        candidates["realization_id"], candidates["fold"] = row.realization_id, row.fold
        samples.append(candidates)
    return pd.concat(samples, ignore_index=True)


def score(model, table):
    table = table.copy()
    probabilities = model.predict_proba(table[FEATURES])
    for label, values in zip(model.classes_, probabilities.T):
        table["score_"+label] = values
    table["priority"] = 1-table.score_background
    table["change_type"] = table[["score_"+k for k in CLASSES[1:]]].idxmax(axis=1).str[6:]
    return table


def train(root, out):
    root, out = Path(root), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    ix, frames, eligible = prepare(root)
    assert ix.groupby("group").fold.nunique().max() == 1
    data = collect(root, ix, frames, eligible)
    train_rows = data[data.fold.isin([2,3,4])]
    positives = train_rows.query("label != 'background'")
    negatives = train_rows.query("label == 'background'")
    train_rows = pd.concat([positives, negatives.sample(min(len(negatives),8*len(positives)),random_state=42)])
    model = RandomForestClassifier(n_estimators=100, min_samples_leaf=6, max_depth=14, n_jobs=2, random_state=42)
    model.fit(train_rows[FEATURES], train_rows.label)
    val = score(model, data[data.fold == 1])
    precision, recall, thresholds = precision_recall_curve(val.label != "background", val.priority)
    f1 = 2*precision*recall / np.maximum(precision+recall, 1e-12)
    threshold = float(thresholds[np.argmax(f1[:-1])])
    joblib.dump((model, threshold),out/"detector.joblib",compress=3)
    test = score(model, data[data.fold == 0])
    flagged = test[test.priority >= threshold]
    actual = test.label != "background"
    metrics = {"training_candidates":len(train_rows), "training_counts":train_rows.label.value_counts().to_dict(),
               "validation_candidates":len(val),"test_candidates":len(test),"test_positive_candidates":int(actual.sum()),
               "test_fields":int(ix.query("fold == 0").stack_id.nunique()), "threshold":threshold,
               "test_average_precision":average_precision_score(actual,test.priority),
               "test_candidate_report":classification_report(test.label,np.where(test.priority>=threshold,test.change_type,"background"),
                    labels=CLASSES,output_dict=True,zero_division=0)}
    eligible_test = eligible[eligible.realization_id.isin(ix[ix.fold==0].realization_id)].query("label != 'background'")
    correct = flagged[flagged.label == flagged.change_type][KEY].drop_duplicates().assign(recovered=True)
    recovery = eligible_test.merge(correct,on=KEY,how="left")
    recovery["recovered"] = recovery.recovered.eq(True)
    recovery.groupby("label").recovered.agg(["sum","count","mean"]).to_csv(out/"frame_recovery.csv")
    recovery.groupby(["label","realization_id","obj_id"]).recovered.any().groupby("label").agg(["sum","count","mean"]).to_csv(out/"object_recovery.csv")
    (out/"metrics.json").write_text(json.dumps(metrics,indent=2))
    return model, threshold, metrics


def scan(root, stack_id, model, threshold):
    root = Path(root)
    ix = pd.read_parquet(root/"index.parquet")
    row = ix[(ix.stack_id==stack_id)&(ix.sample_type=="real_stack")].iloc[0]
    frames = pd.read_parquet(root/"spherex/frames.parquet")
    meta = frames[(frames.stack_id==stack_id)&frames.qc_pass].set_index("frame_index").sort_index()
    peaks = proposals(root/row.path,meta,False)
    if peaks.empty:
        return pd.DataFrame(columns=["frame_index","x","y","priority","change_type","mjd_mid","detector"])
    ranked = score(model,peaks).join(meta[["mjd_mid","detector"]],on="frame_index")
    return ranked[ranked.priority>=threshold].sort_values("priority",ascending=False)


In [3]:
out = Path('results')
model, threshold, metrics = train(dataset, out)
print(json.dumps(metrics, indent=2))
display(pd.read_csv(out/'frame_recovery.csv'), pd.read_csv(out/'object_recovery.csv'))

{
  "training_candidates": 122283,
  "training_counts": {
    "background": 108696,
    "motion": 8741,
    "variable": 4620,
    "transient": 226
  },
  "validation_candidates": 153256,
  "test_candidates": 117825,
  "test_positive_candidates": 4950,
  "test_fields": 8,
  "threshold": 0.5919007184718609,
  "test_average_precision": 0.5007703900710903,
  "test_candidate_report": {
    "background": {
      "precision": 0.9775958790530289,
      "recall": 0.9819003322259137,
      "f1-score": 0.9797433778127445,
      "support": 112875.0
    },
    "motion": {
      "precision": 0.4685746352413019,
      "recall": 0.5475409836065573,
      "f1-score": 0.5049894163894769,
      "support": 3050.0
    },
    "variable": {
      "precision": 0.7120359955005624,
      "recall": 0.3503043718871057,
      "f1-score": 0.46958456973293766,
      "support": 1807.0
    },
    "transient": {
      "precision": 0.0,
      "recall": 0.0,
      "f1-score": 0.0,
      "support": 93.0
    },
    "accura

## Scan unmodified SPHEREx images
Each row gives a frame, position on the aligned grid, date, detector, class scores and review priority. Higher priority means stronger model evidence for a change. The score is uncalibrated.

In [4]:
index = pd.read_parquet(dataset/'index.parquet')
rows = index.query("mission == 'spherex' and sample_type == 'real_stack'")
model, threshold = joblib.load(out/'detector.joblib')
real = pd.concat([scan(dataset, sid, model, threshold).assign(stack_id=sid) for sid in rows.stack_id],ignore_index=True)
real.to_csv(out/'real_candidates.csv.gz',index=False)
display(real.nlargest(10,'priority'))

       peak      flux_z  reference_z      diff_z         dip        rise  centroid_shift  flux_contrast  valid_fraction  wave_gap  frame_index  x   y  score_background  score_motion  score_transient  score_variable  priority change_type      mjd_mid  detector                         stack_id
  87.041786         NaN          NaN   87.041786  -66.392258   87.041786        0.935622            NaN            1.00  0.035461           56 63  63          0.020189      0.977954         0.000821        0.001037  0.979811      motion 61314.190606         2           spherex:epsilon-indi-b
 446.016327 3365.808838  2003.520020  446.016327 -116.120445  446.016327        0.898647       0.253670            1.00  0.091748            8 64  64          0.032627      0.961402         0.000762        0.005209  0.967373      motion 60795.086340         3           spherex:epsilon-indi-b
2594.362305 9037.453125    55.083099 2594.362305   19.046432 2594.362305        0.998963       0.987775            1.00  

In [5]:
"""Compare blind real-image detections with the five named catalogue targets."""
from pathlib import Path
import numpy as np
import pandas as pd

TARGETS = {"barnards-star":"motion","epsilon-indi-b":"motion","wise0855":"motion",
           "rw-aurigae":"variable","r-draconis":"variable"}


def audit_targets(root, candidates):
    root = Path(root)
    index = pd.read_parquet(root/"index.parquet").query("mission == 'spherex' and sample_type == 'real_stack'")
    source = pd.read_parquet(root/"spherex/sources.parquet").query("is_target")
    frames = pd.read_parquet(root/"spherex/frames.parquet").query("qc_pass")
    rows = []
    for field, expected in TARGETS.items():
        row = index[index.field == field].iloc[0]
        sid = row.stack_id
        targets = source[source.stack_id == sid]
        columns = ["stack_id","src_id","frame_index","x","y","saturated","contaminated","aperture_valid_frac","snr"]
        tracks = pd.read_parquet(root/"spherex/tracks.parquet",columns=columns,filters=[("stack_id","=",sid)])
        tracks = tracks[tracks.src_id.isin(targets.src_id) & tracks.frame_index.isin(frames[frames.stack_id==sid].frame_index)]
        usable = tracks[(~tracks.saturated)&(~tracks.contaminated)&(tracks.aperture_valid_frac>=0.95)&(tracks.snr>=5)]
        hits = set()
        nearest = candidates[(candidates.stack_id==sid)&(candidates.change_type==expected)]
        for t in usable.itertuples():
            same = nearest[nearest.frame_index==t.frame_index]
            if np.any(np.hypot(same.x-t.x,same.y-t.y)<=2):
                hits.add(t.frame_index)
        rows.append(dict(target=field,split=row.split,expected=expected,
                         targets=len(targets),qc_frames=frames[frames.stack_id==sid].frame_index.nunique(),
                         saturated_frames=tracks.loc[tracks.saturated,"frame_index"].nunique(),
                         usable_frames=usable.frame_index.nunique(),matched_frames=len(hits)))
    return pd.DataFrame(rows)


In [6]:
targets = audit_targets(dataset, real)
targets.to_csv(out/'named_targets.csv', index=False)
display(targets)

        target split expected  targets  qc_frames  saturated_frames  usable_frames  matched_frames
 barnards-star train   motion        1         58                58              0               0
epsilon-indi-b   val   motion        1         59                12             42              13
      wise0855  test   motion        1         60                 8             29               0
    rw-aurigae   val variable        2         59                57              2               0
    r-draconis  test variable        1         56                56              0               0
